# cAPTure: four-model development comparison

This notebook compares the completed seed-42 Edge MLP, EdgeGRU, StaticGNN, and ST-GNN jobs across both development folds. It verifies immutable artifacts and exact OOF alignment, then reports ranking, operational, early-warning, scenario, step, and resource results.

This is an exploratory partial comparison. It performs no training and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`. A topology-centered claim remains unavailable until the matched ST-GNN control without GAT message passing is complete.

## 1. Prepare the Colab environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_comparison import (
    run_graph_four_model_comparison,
    validate_graph_four_model_comparison,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this evaluation.")

## 2. Bind the completed training jobs

The comparison is bound to the authorized graph-training run. All eight jobs listed below must already be complete.

In [ ]:
BINDING_RUN_ID = "20260929T144234_934044Z_graph_training_binding"
BINDING_DIR = DRIVE_ROOT / "graph_training_binding_runs" / BINDING_RUN_ID
TRAINING_ROOT = DRIVE_ROOT / "graph_training_runs" / BINDING_RUN_ID
TRAINING_CONFIG_PATH = PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
TERMINAL_POLICY_PATH = PROJECT_ROOT / "configs/capture_early_warning_audit_v1.yaml"

MODEL_NAMES = ["edge_mlp", "edge_gru", "static_gnn", "st_gnn"]
FOLDS = ["A", "B"]
required_paths = [
    BINDING_DIR, TRAINING_ROOT, TRAINING_CONFIG_PATH, MANIFEST_PATH,
    TERMINAL_POLICY_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required comparison inputs are missing: {missing}")

inventory = []
for model_name in MODEL_NAMES:
    for fold in FOLDS:
        job_id = f"{model_name}__fold_{fold}"
        completion_path = TRAINING_ROOT / job_id / "completion.json"
        completion = (
            json.loads(completion_path.read_text(encoding="utf-8"))
            if completion_path.is_file() else {}
        )
        inventory.append({
            "job_id": job_id,
            "completion_exists": completion_path.is_file(),
            "status": completion.get("status"),
            "best_epoch_count": completion.get("best_epoch_count"),
            "held_out_scenarios_accessed": completion.get("held_out_scenarios_accessed"),
        })
inventory_table = pd.DataFrame(inventory)
display(inventory_table)
if not inventory_table["completion_exists"].all() or not (inventory_table["status"] == "complete").all():
    raise RuntimeError("All eight model/fold jobs must be complete before comparison.")

## 3. Create or verify the immutable comparison

The first run verifies every source checksum and compares packet keys, timestamps, labels, steps, and sequence IDs across models before computing metrics. Set `RUN_COMPARISON=True` only after reviewing the inventory. A normal rerun with the same run ID validates and reuses the completed report.

In [ ]:
COMPARISON_RUN_ID = None  # Set this to reuse a completed comparison.
RUN_COMPARISON = False
if COMPARISON_RUN_ID is None:
    COMPARISON_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_four_model_comparison"
    )
OUTPUT_DIR = DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID
print("Comparison run ID:", COMPARISON_RUN_ID)
print("Comparison output:", OUTPUT_DIR)

if OUTPUT_DIR.exists():
    comparison_report = validate_graph_four_model_comparison(
        output_dir=OUTPUT_DIR,
        training_root=TRAINING_ROOT,
        binding_dir=BINDING_DIR,
        training_config_path=TRAINING_CONFIG_PATH,
        manifest_path=MANIFEST_PATH,
        terminal_policy_path=TERMINAL_POLICY_PATH,
    )
elif RUN_COMPARISON:
    comparison_report = run_graph_four_model_comparison(
        training_root=TRAINING_ROOT,
        binding_dir=BINDING_DIR,
        training_config_path=TRAINING_CONFIG_PATH,
        manifest_path=MANIFEST_PATH,
        terminal_policy_path=TERMINAL_POLICY_PATH,
        output_dir=OUTPUT_DIR,
        batch_size=250_000,
    )
else:
    comparison_report = None
    print("Set RUN_COMPARISON=True to start the read-only OOF evaluation.")

## 4. Ranking comparison

The primary aggregation first averages scenarios within each outer fold and then gives equal weight to the two folds. The flat mean across all five scenarios is retained only as a diagnostic.

In [ ]:
if comparison_report is None:
    raise RuntimeError("Run or load the comparison before displaying results.")
ranking_summary = pd.DataFrame(comparison_report["tables"]["ranking_summary"])
ranking_by_scenario = pd.DataFrame(
    comparison_report["tables"]["ranking_by_scenario"]
)
display(ranking_summary.set_index("model_display")[[
    "average_precision", "roc_auc",
    "flat_average_precision_diagnostic", "flat_roc_auc_diagnostic",
]])
display(ranking_by_scenario.set_index(["model_display", "scenario"])[[
    "fold", "packets", "normal_packets", "attack_packets",
    "average_precision", "roc_auc",
]])

## 5. Primary operational comparison

Each model receives its own development-OOF threshold under the frozen budget of one false-alert window per hour. A timely iteration requires the first correct alert window to close strictly before the final malicious packet. Preterminal coverage requires a correct alert strictly before the first declared terminal-action packet. Because threshold calibration and reporting reuse the same development OOF predictions, these are screening estimates rather than confirmatory estimates.

In [ ]:
operational_summary = pd.DataFrame(
    comparison_report["tables"]["primary_operational_summary"]
)
operational_by_scenario = pd.DataFrame(
    comparison_report["tables"]["primary_operational_by_scenario"]
)
display(operational_summary.set_index("model_display")[[
    "threshold", "worst_fold_false_alert_windows_per_hour",
    "packet_precision", "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate",
    "timely_iteration_rate", "early_before_terminal_action",
    "median_seconds_before_iteration_end",
    "median_seconds_before_terminal_action",
]])
display(operational_by_scenario.set_index(["model_display", "scenario"])[[
    "fold", "false_alert_windows_per_hour", "packet_precision",
    "packet_recall", "packet_f1", "packet_f2",
    "iterations", "detected_iterations",
    "timely_iterations", "timely_iteration_rate",
    "early_before_terminal_action", "seconds_before_terminal_action",
]])

## 6. False-alert-budget sensitivity

One false-alert window per hour remains the primary budget. The stricter one-per-12-hours budget and the more permissive one-per-five-minutes budget were frozen before this comparison and are reported as sensitivity analyses. They must not be used to replace the primary budget after inspecting results.

In [ ]:
budget_sensitivity = pd.DataFrame(
    comparison_report["tables"]["budget_sensitivity_summary"]
)
budget_sensitivity_by_scenario = pd.DataFrame(
    comparison_report["tables"]["budget_sensitivity_by_scenario"]
)
display(budget_sensitivity.set_index(["model_display", "budget"])[[
    "target_false_alert_windows_per_hour",
    "worst_fold_false_alert_windows_per_hour", "threshold",
    "packet_precision", "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate", "timely_iteration_rate",
    "early_before_terminal_action",
]])
display(budget_sensitivity_by_scenario.set_index(
    ["model_display", "budget", "scenario"]
)[[
    "fold", "false_alert_windows_per_hour", "packet_precision",
    "packet_recall", "packet_f1", "packet_f2",
    "sequence_detection_rate", "timely_iteration_rate",
]])

## 7. Predeclared pairwise contrasts

`StaticGNN - Edge MLP` is the current aggregation/message-passing signal without recurrent memory. `ST-GNN - EdgeGRU` is the temporal comparison, but it does not perfectly isolate GAT because the model families still differ. The pending matched ST-GNN control disables GATv2 layers while retaining endpoint aggregation and per-node memory.

In [ ]:
pairwise = pd.DataFrame(comparison_report["tables"]["pairwise_comparisons"])
display(pairwise.set_index(["comparison", "metric"])[[
    "candidate", "control", "candidate_hierarchical_macro",
    "control_hierarchical_macro", "hierarchical_macro_delta",
    "frozen_practical_margin", "scenarios_with_positive_delta",
    "scenarios_with_zero_delta", "scenarios_with_negative_delta",
]])

## 8. Step breakdown and resource cost

In [ ]:
step_metrics = pd.DataFrame(comparison_report["tables"]["primary_step_metrics"])
resource_summary = pd.DataFrame(comparison_report["tables"]["resource_summary"])
resource_by_job = pd.DataFrame(comparison_report["tables"]["resource_by_job"])
display(step_metrics.set_index(["model_display", "scenario", "attack_step"])[[
    "fold", "iterations", "detected_iterations", "timely_iterations",
    "sequence_detection_rate", "timely_iteration_rate",
    "median_timely_lead_seconds",
]])
display(resource_summary.set_index("model_display")[[
    "best_epoch_count_fold_A", "best_epoch_count_fold_B", "parameters",
    "total_wall_seconds", "seconds_per_million_packets",
    "maximum_peak_host_rss_gib", "maximum_peak_cuda_allocated_gib",
]])
display(resource_by_job.set_index(["model_display", "fold"]))

## 9. Review the saved English report

The generated `review.md` is descriptive and explicitly preserves the pending-ablation boundary. It does not automatically approve topology as the central proposal component.

In [ ]:
display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))
print("Comparison status:", comparison_report["status"])
print("Models compared:", comparison_report["models_compared"])
print("Models pending:", comparison_report["models_pending"])
print("Topology claim authorized:", comparison_report["topology_claim_authorized"])
print("Model training performed:", comparison_report["model_training_performed"])
print("Held-out scenarios accessed:", comparison_report["held_out_scenarios_accessed"])
print("Saved report:", OUTPUT_DIR / "comparison_report.json")
print("Saved review:", OUTPUT_DIR / "review.md")